# Comparación de resultados de la semana del 15

Compara los experimentos guardados en Drive: visión completa, 3×3, 5×5, 7×7, 5×5 con sampler de 1 y 5×5 con sampler de 1 y más entropía. También detecta otras pruebas que se hayan guardado en `galleta/semana15/`.

**Abre este notebook en una sesión nueva de CPU.** Solo lee métricas y configuraciones; no abre modelos ni buffers, no inicia entrenamientos ni modifica los runs. Los dos entrenamientos actuales pueden seguir corriendo en sus sesiones.

**Primera vez:** ejecuta las secciones 1 a 4. **Mientras entrenan:** vuelve a ejecutar la sección 4 para releer los archivos y regenerar los gráficos. Drive puede mostrar datos con atraso; si no se actualiza, puedes volver a montar Drive en esta sesión comparadora con `force_remount=True` en la sección 1.

Verás una tabla de avance y resultados, curvas de evaluación, una comparación específica de 5×5, exploración, recompensa imaginada y entropía. También se exportan PNG, PDF y CSV a Drive y se prepara un ZIP para descargar.

**Cómo leerlo:** la última evaluación puede corresponder a distintos pasos en cada run. El gráfico «Evaluación al mismo paso» usa el último paso que tiene evaluación en todos los runs que ya tengan alguna, sin interpolar. Los runs sin evaluación se muestran como ausentes. El suavizado solo se aplica a episodios de entrenamiento. Cada curva representa una corrida; estas comparaciones no son promedios de varias semillas ni demuestran por sí solas un efecto general.


## 1. Montar Drive

Usa la misma cuenta donde guardaste los entrenamientos. Mantén este notebook en CPU.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## 2. Traer el comparador

Instala solo lo necesario para leer y dibujar resultados. No hace falta instalar Dreamer ni cambiar la versión de Python de Colab.


In [ ]:
%%bash
set -e
if [ -d /content/r2dreamer-cookie ]; then
  git -C /content/r2dreamer-cookie pull --ff-only
else
  git clone https://github.com/FabriRandon/r2dreamer-cookie.git /content/r2dreamer-cookie
fi


In [ ]:
%pip -q install matplotlib pandas PyYAML


## 3. Elegir las carpetas y el suavizado

La detección automática busca las carpetas de los runs dentro de `RAIZ`. El caso de visión completa se guardaba en `semana8` y por eso se incluye como referencia adicional. Una prueba que aún no exista se omite de los gráficos, sin asignarle resultados cero.

`VENTANA` controla cuántas partidas recientes promedia la curva de entrenamiento; las evaluaciones se muestran sin suavizar. Los resultados generados quedan en una carpeta propia, `comparacion`, fuera de las carpetas de entrenamiento.


In [ ]:
from pathlib import Path

RAIZ = Path("/content/drive/MyDrive/galleta/semana15")
EXTRAS = [Path("/content/drive/MyDrive/galleta/semana8/cookie_full_hall5_s0_base")]
EXCLUIR = []  # nombres de carpetas de runs que quieras omitir
VENTANA = 20
SALIDA = RAIZ / "comparacion"

ESPERADOS = {
    "Visión completa": EXTRAS[0],
    "3×3 original": RAIZ / "cookie_partial_hall5_view3_s0_base",
    "5×5 original": RAIZ / "cookie_partial_hall5_view5_s0_base",
    "7×7 original": RAIZ / "cookie_partial_hall5_view7_s0_base",
    "5×5 sampler 1": RAIZ / "cookie_partial_hall5_view5_s0_base_sampler1",
    "5×5 sampler 1 + entropía": RAIZ / "cookie_partial_hall5_view5_s0_base_sampler1_entropia10x",
}
print("Lectura:", RAIZ)
print("Gráficos y tablas:", SALIDA)


## 4. Leer los datos y comparar

**Reejecuta esta celda para actualizar mientras los otros notebooks siguen entrenando.** La lectura conserva los datos disponibles en ese momento; puede haber atraso de sincronización de Drive. Una línea aún incompleta se ignora hasta la siguiente lectura.

«Checkpoint al objetivo» significa que `status.json` confirma un guardado al paso objetivo. «Parcial / final sin confirmar» puede ser un run activo, una sesión detenida o un guardado final todavía no sincronizado; el comparador no controla ni inspecciona esos procesos.

La tabla muestra cuántas partidas se usaron por evaluación según la configuración guardada. El promedio de entrenamiento corresponde a las últimas partidas disponibles. La mejor evaluación es el máximo histórico, no una medida final independiente. La recompensa imaginada puede cambiar por el sampler: evalúa el aprendizaje mirando las galletas en evaluación.


In [ ]:
import importlib
import sys

import pandas as pd
from IPython.display import Image, Markdown, display

REPO = "/content/r2dreamer-cookie"
if REPO not in sys.path:
    sys.path.insert(0, REPO)
import compare_runs
importlib.reload(compare_runs)


def tabla(rows, columnas=None):
    frame = pd.DataFrame(rows)
    if columnas is not None:
        frame = frame[columnas]
    display(frame.astype(object).where(pd.notna(frame), "—"))


def actualizar():
    display(Markdown("### Carpetas esperadas"))
    tabla([{"Prueba": nombre,
            "Datos": "Con métricas" if (ruta / "metrics.jsonl").exists() else "Sin métricas todavía",
            "Carpeta": str(ruta)} for nombre, ruta in ESPERADOS.items()])
    runs = [r for r in compare_runs.discover(RAIZ, EXTRAS) if r.path.name not in EXCLUIR]
    if not runs:
        print("No se encontraron métricas. Revisa la cuenta de Drive y las rutas de la sección 3.")
        return None
    resultado = compare_runs.generate_report(runs, SALIDA, VENTANA)
    display(Markdown(f"### Resultados disponibles · lectura UTC {resultado['read_at']}"))
    tabla(resultado["summary"], ["Experimento", "Último paso en métricas", "Pasos objetivo",
          "Último checkpoint", "Estado guardado", "Paso última evaluación", "Última evaluación",
          "Mejor evaluación", "Paso primera evaluación con galletas", "Promedio entrenamiento reciente"])
    display(Markdown("### Configuración guardada de cada prueba"))
    tabla(resultado["summary"], ["Carpeta", "Tarea", "Semilla", "Sampler", "Coeficiente entropía",
          "Partidas por evaluación", "Evaluaciones", "Avisos"])
    if resultado["common"]:
        display(Markdown(f"### Comparación al mismo paso: {resultado['common_step']:,}"))
        tabla(resultado["common"])
        sin_eval = [r.path.name for r in runs if not r.points("episode/eval_score")]
        if sin_eval:
            print("Aún sin evaluación; fuera de esta comparación:", ", ".join(sin_eval))
    else:
        print("Todavía no hay un paso de evaluación compartido entre los runs con evaluaciones.")
    for imagen in resultado["images"]:
        display(Image(filename=str(imagen)))
    print("Archivos PNG, PDF y CSV guardados en:", resultado["out"])
    print("ZIP listo para descargar:", resultado["archive"])
    return resultado


resultado = actualizar()


## 5. Descargar los resultados (opcional)

Cambia `DESCARGAR_ZIP` a `True` y ejecuta la celda para descargar los gráficos en PNG/PDF, las tablas CSV y el registro de la lectura. Los CSV usan codificación compatible con Excel. El archivo `curvas.csv` contiene los puntos registrados, sin suavizado ni interpolación. Los mismos archivos ya quedan en Drive en la carpeta `comparacion`.


In [ ]:
DESCARGAR_ZIP = False

if DESCARGAR_ZIP:
    from google.colab import files
    if globals().get("resultado") is None:
        raise RuntimeError("Primero ejecuta la sección 4 y verifica que haya resultados.")
    files.download(str(resultado["archive"]))
